In [0]:
%python
from pyspark.sql.functions import current_timestamp, col,lit

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", "/Volumes/associate_assignment/default/raw/customer_volume/master_customer/schema/")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load("/Volumes/associate_assignment/default/raw/customer_volume/master_customer/landing/")
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("updated_timestamp", current_timestamp())
        .withColumn("effective_from",current_timestamp())
        .withColumn("effective_to",lit("9999-12-31"))
        .withColumn("current_flag",lit("Y"))
        .withColumn("source_file", col("_metadata.file_path"))
)

(
    df.writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/associate_assignment/default/raw/customer_volume/master_customer/checkpoint/")
        .trigger(availableNow=True)
        .toTable("associate_assignment.default.bronze_master_customer")
)